# Data Quality Tests for Wheelie Data Warehouse

This notebook contains data quality tests for all dimension tables and bridge tables.
Each test validates uniqueness constraints and referential integrity.

In [ ]:
# ==============================================================================
# TEST CONFIGURATION & HELPER FUNCTIONS
# ==============================================================================
import logging
from pyspark.sql.functions import col, xxhash64

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger("data_quality_tests")

def get_table(table_name: str):
    """Helper to load table from data warehouse."""
    return spark.table(f"wheelie.gold.{table_name}")

def assert_uniqueness(table_name: str, column_name: str):
    """Assert that a column has all unique values."""
    df = get_table(table_name)
    total_count = df.count()
    distinct_count = df.select(column_name).distinct().count()
    duplicates_count = total_count - distinct_count

    if duplicates_count > 0:
        duplicates = df.groupBy(column_name).count().filter(col("count") > 1)
        logger.error(f"Duplicate values found in {table_name}.{column_name}:")
        duplicates.show(100)

    assert total_count == distinct_count, \
        f"{table_name}.{column_name}: Found {duplicates_count} duplicate(s). Expected all {total_count} values to be unique."


def assert_uniqueness_where(table_name: str, column_name: str, filter_expr):
    """Assert uniqueness for a filtered subset of a table."""
    df = get_table(table_name).filter(filter_expr)
    total_count = df.count()
    distinct_count = df.select(column_name).distinct().count()
    duplicates_count = total_count - distinct_count

    if duplicates_count > 0:
        duplicates = df.groupBy(column_name).count().filter(col("count") > 1)
        logger.error(f"Duplicate values found in {table_name}.{column_name} (filtered):")
        duplicates.show(100)

    assert total_count == distinct_count, \
        f"{table_name}.{column_name} (filtered): Found {duplicates_count} duplicate(s). Expected all {total_count} values to be unique."

def assert_referential_integrity(source_table: str, source_col: str, target_table: str, target_col: str):
    """Assert that all values in source column exist in target column (excludes nulls)."""
    source_df = get_table(source_table)
    target_df = get_table(target_table)

    # Filter out nulls from source before checking
    missing_keys = source_df.filter(col(source_col).isNotNull()) \
        .select(col(source_col).alias("key")) \
        .distinct() \
        .join(
            target_df.select(col(target_col).alias("key")),
            "key",
            "left_anti"
        )

    missing_count = missing_keys.count()
    total_count = source_df.filter(col(source_col).isNotNull()).select(source_col).distinct().count()

    if missing_count > 0:
        logger.error(f"Missing references from {source_table}.{source_col} to {target_table}.{target_col}:")
        logger.error("Missing keys:")
        missing_keys.show(100, truncate=False)

        # Show full rows from source table for debugging
        logger.error(f"\nFull rows from {source_table} with missing keys:")
        source_df.join(missing_keys, source_df[source_col] == missing_keys["key"]) \
            .drop("key") \
            .show(100, truncate=False)

        # Post-action diagnostics: column types can explain hash/key mismatches
        source_type = dict(source_df.dtypes).get(source_col)
        target_type = dict(target_df.dtypes).get(target_col)
        logger.error(
            f"Type check: {source_table}.{source_col}={source_type}, "
            f"{target_table}.{target_col}={target_type}"
        )

        # Post-action diagnostics: surface business keys for known surrogate joins
        if source_table == "fact_rental" and source_col == "customer_key" and target_table == "dim_customer":
            try:
                rental_bronze = spark.table("wheelie.bronze.rental")
                missing_customer_ids = (
                    rental_bronze.select(col("customer_id"))
                    .withColumn("customer_key", xxhash64(col("customer_id")))
                    .join(missing_keys.withColumnRenamed("key", "customer_key"), "customer_key", "inner")
                    .select("customer_id")
                    .distinct()
                )
                logger.error("Missing customer_id values (from bronze rental):")
                missing_customer_ids.show(50, truncate=False)
            except Exception as e:
                logger.error(f"Post-action diagnostics failed: {str(e)}")

    assert missing_count == 0, \
        f"Referential integrity violation: {missing_count} out of {total_count} keys in {source_table}.{source_col} do not exist in {target_table}.{target_col}"
    logger.info("=" * 70)

def assert_not_null(table_name: str, column_name: str):
    """Assert that a column has no NULL values (required field completeness)."""
    df = get_table(table_name)
    null_count = df.filter(col(column_name).isNull()).count()
    total_count = df.count()

    if null_count > 0:
        logger.error(f"NULL values found in {table_name}.{column_name}:")
        logger.error(f"  {null_count} out of {total_count} rows have NULL values")

    assert null_count == 0, \
        f"{table_name}.{column_name}: Found {null_count} NULL value(s). Expected all {total_count} values to be NOT NULL."

logger.info("DATA QUALITY TEST FRAMEWORK LOADED")
logger.info("=" * 70)


In [ ]:
# ==============================================================================
# TEST DEFINITIONS: DIMENSIONS
# ==============================================================================

def test_dim_date_key_unique():
    """Test that date_key is unique in dim_date."""
    assert_uniqueness("dim_date", "date_key")

def test_dim_service_date_key_unique():
    """Test that service_date_key is unique in dim_service_date."""
    assert_uniqueness("dim_service_date", "service_date_key")

def test_dim_rental_date_key_unique():
    """Test that rental_date_key is unique in dim_rental_date."""
    assert_uniqueness("dim_rental_date", "rental_date_key")

def test_dim_return_date_key_unique():
    """Test that return_date_key is unique in dim_return_date."""
    assert_uniqueness("dim_return_date", "return_date_key")

def test_dim_payment_date_key_unique():
    """Test that payment_date_key is unique in dim_payment_date."""
    assert_uniqueness("dim_payment_date", "payment_date_key")

def test_dim_payment_deadline_date_key_unique():
    """Test that payment_deadline_date_key is unique in dim_payment_deadline_date."""
    assert_uniqueness("dim_payment_deadline_date", "payment_deadline_date_key")

def test_dim_staff_staff_key_unique():
    """Test that staff_key is unique in dim_staff."""
    assert_uniqueness("dim_staff", "staff_key")

def test_dim_manager_manager_key_unique():
    """Test that manager_key is unique in dim_manager."""
    assert_uniqueness("dim_manager", "manager_key")

def test_dim_manager_staff_id_unique():
    """Test that staff_id is unique in dim_manager."""
    assert_uniqueness("dim_manager", "staff_id")

def test_dim_store_store_key_unique():
    """Test that store_key is unique for current records in dim_store."""
    assert_uniqueness_where("dim_store", "store_key", col("is_current") == True)

def test_dim_store_store_id_unique():
    """Test that store_id is unique for current records in dim_store."""
    assert_uniqueness_where("dim_store", "store_id", col("is_current") == True)

def test_dim_car_car_key_unique():
    """Test that car_key is unique in dim_car."""
    assert_uniqueness("dim_car", "car_key")

def test_dim_customer_customer_key_unique():
    """Test that customer_key is unique in dim_customer."""
    assert_uniqueness("dim_customer", "customer_key")

def test_dim_customer_customer_id_unique():
    """Test that customer_id is unique in dim_customer."""
    assert_uniqueness("dim_customer", "customer_id")

def test_dim_equipment_equipment_key_unique():
    """Test that equipment_key is unique in dim_equipment."""
    assert_uniqueness("dim_equipment", "equipment_key")

logger.info("✅ Dimension test definitions loaded")


In [ ]:
# ==============================================================================
# TEST DEFINITIONS: BRIDGE TABLES
# ==============================================================================

def test_bridge_car_equipment_car_key_unique():
    """Test that car_key is unique in bridge_car_equipment."""
    assert_uniqueness("bridge_car_equipment", "car_key")

def test_bridge_equipment_group_equipment_group_key_exists():
    """Test that all equipment_group_key values exist in bridge_car_equipment."""
    assert_referential_integrity(
        source_table="bridge_equipment_group_equipment",
        source_col="equipment_group_key",
        target_table="bridge_car_equipment",
        target_col="equipment_group_key"
    )

def test_bridge_equipment_group_equipment_key_exists():
    """Test that all equipment_key values exist in dim_equipment."""
    assert_referential_integrity(
        source_table="bridge_equipment_group_equipment",
        source_col="equipment_key",
        target_table="dim_equipment",
        target_col="equipment_key"
    )

def test_bridge_staff_hierarchy_staff_key_exists():
    """Test that all staff_key values exist in dim_staff."""
    assert_referential_integrity(
        source_table="bridge_staff_hierarchy",
        source_col="staff_key",
        target_table="dim_staff",
        target_col="staff_key"
    )

def test_bridge_staff_hierarchy_manager_key_exists():
    """Test that all manager_key values exist in dim_manager."""
    assert_referential_integrity(
        source_table="bridge_staff_hierarchy",
        source_col="manager_key",
        target_table="dim_manager",
        target_col="manager_key"
    )

logger.info("✅ Bridge table test definitions loaded")


In [ ]:
# ==============================================================================
# TEST DEFINITIONS: FACT TABLES
# ==============================================================================

def test_fact_service_key_unique():
    """Test that service_key is unique in fact_service."""
    assert_uniqueness("fact_service", "service_key")

def test_fact_service_car_key_exists():
    """Test that all car_key values in fact_service exist in dim_car."""
    assert_referential_integrity("fact_service", "car_key", "dim_car", "car_key")

def test_fact_rental_key_unique():
    """Test that rental_key is unique in fact_rental."""
    assert_uniqueness("fact_rental", "rental_key")

def test_fact_rental_customer_key_exists():
    """Test that all customer_key values in fact_rental exist in dim_customer."""
    assert_referential_integrity("fact_rental", "customer_key", "dim_customer", "customer_key")

def test_fact_rental_car_key_exists():
    """Test that all car_key values in fact_rental exist in dim_car."""
    assert_referential_integrity("fact_rental", "car_key", "dim_car", "car_key")

def test_fact_rental_staff_key_exists():
    """Test that all staff_key values in fact_rental exist in dim_staff."""
    assert_referential_integrity("fact_rental", "staff_key", "dim_staff", "staff_key")

def test_fact_rental_store_key_exists():
    """Test that all store_key values in fact_rental exist in dim_store."""
    assert_referential_integrity("fact_rental", "store_key", "dim_store", "store_key")

def test_fact_rental_date_keys_exist():
    """Test that all date keys in fact_rental exist in their respective date dimensions (excluding nulls)."""
    # Test rental_date_key -> dim_rental_date
    assert_referential_integrity("fact_rental", "rental_date_key", "dim_rental_date", "rental_date_key")

    # Test return_date_key -> dim_return_date (nulls filtered automatically)
    assert_referential_integrity("fact_rental", "return_date_key", "dim_return_date", "return_date_key")

    # Test payment_date_key -> dim_payment_date (nulls filtered automatically)
    assert_referential_integrity("fact_rental", "payment_date_key", "dim_payment_date", "payment_date_key")

    # Test payment_deadline_date_key -> dim_payment_deadline_date
    assert_referential_integrity("fact_rental", "payment_deadline_date_key", "dim_payment_deadline_date", "payment_deadline_date_key")
logger.info("✅ Fact table test definitions loaded")


In [ ]:
# ==============================================================================
# TEST DEFINITIONS: REQUIRED FIELD COMPLETENESS (NOT NULL)
# ==============================================================================

def test_dim_customer_required_fields():
    """Test that required fields in dim_customer are NOT NULL."""
    assert_not_null("dim_customer", "customer_id")
    assert_not_null("dim_customer", "customer_first_name")
    assert_not_null("dim_customer", "customer_last_name")

def test_dim_staff_required_fields():
    """Test that required fields in dim_staff are NOT NULL."""
    assert_not_null("dim_staff", "staff_id")
    assert_not_null("dim_staff", "staff_first_name")
    assert_not_null("dim_staff", "staff_last_name")
    assert_not_null("dim_staff", "staff_email")

def test_dim_store_required_fields():
    """Test that required fields in dim_store are NOT NULL."""
    assert_not_null("dim_store", "store_id")
    assert_not_null("dim_store", "city")
    assert_not_null("dim_store", "country")

def test_dim_car_required_fields():
    """Test that required fields in dim_car are NOT NULL."""
    assert_not_null("dim_car", "inventory_id")
    assert_not_null("dim_car", "rental_rate")
    assert_not_null("dim_car", "fuel_type")

def test_fact_rental_required_fields():
    """Test that required fields in fact_rental are NOT NULL."""
    assert_not_null("fact_rental", "rental_id")
    assert_not_null("fact_rental", "customer_key")
    assert_not_null("fact_rental", "car_key")
    assert_not_null("fact_rental", "staff_key")
    assert_not_null("fact_rental", "store_key")
    assert_not_null("fact_rental", "rental_date")
    assert_not_null("fact_rental", "rental_rate")

def test_fact_service_required_fields():
    """Test that required fields in fact_service are NOT NULL."""
    assert_not_null("fact_service", "service_id")
    assert_not_null("fact_service", "car_key")
    assert_not_null("fact_service", "service_date_key")

logger.info("✅ Required field completeness test definitions loaded")


In [ ]:
# ==============================================================================
# TEST DEFINITIONS: DATA RECONCILIATION (ETL INTEGRITY)
# ==============================================================================

def test_fact_rental_payment_reconciliation():
    """Test that sum of payment_amount in fact_rental matches sum of amount in source payment table."""
    from pyspark.sql.functions import sum as spark_sum, coalesce
    from decimal import Decimal

    # Get sum from source payment table (bronze layer)
    payment_bronze = spark.table("wheelie.bronze.payment")
    source_payment_sum = payment_bronze.agg(
        coalesce(spark_sum("amount"), lit(0)).alias("total")
    ).collect()[0]["total"]

    # Get sum from data warehouse fact_rental
    fact_rental = get_table("fact_rental")
    warehouse_payment_sum = fact_rental.agg(
        coalesce(spark_sum("payment_amount"), lit(0)).alias("total")
    ).collect()[0]["total"]

    # Convert to Decimal for precise arithmetic (handles large sums like 40M+ properly)
    source_decimal = Decimal(str(source_payment_sum)) if source_payment_sum is not None else Decimal(0)
    warehouse_decimal = Decimal(str(warehouse_payment_sum)) if warehouse_payment_sum is not None else Decimal(0)

    # Log values for transparency
    logger.info(f"Source payment.amount sum: {source_decimal:,.2f}")
    logger.info(f"Warehouse fact_rental.payment_amount sum: {warehouse_decimal:,.2f}")

    # Calculate difference using Decimal arithmetic (no float precision loss)
    difference = abs(source_decimal - warehouse_decimal)
    difference_pct = (difference / source_decimal * 100) if source_decimal > 0 else Decimal(0)

    logger.info(f"Difference: {difference:,.2f} ({difference_pct:.4f}%)")

    # Allow for small rounding differences (< 0.01)
    tolerance = Decimal("0.01")

    assert difference < tolerance, \
        f"Payment reconciliation failed: Source sum = {source_decimal:,.2f}, Warehouse sum = {warehouse_decimal:,.2f}, Difference = {difference:,.2f}"

logger.info("✅ Data reconciliation test definitions loaded")


In [ ]:
# ==============================================================================
# RUN ALL TESTS
# ==============================================================================

logger.info("\n" + "=" * 70)
logger.info("EXECUTING ALL DATA QUALITY TESTS")
logger.info("=" * 70 + "\n")

# Collect all test functions
test_functions = [
    # Dimension tests
    ("dim_date", test_dim_date_key_unique),
    ("dim_service_date", test_dim_service_date_key_unique),
    ("dim_rental_date", test_dim_rental_date_key_unique),
    ("dim_return_date", test_dim_return_date_key_unique),
    ("dim_payment_date", test_dim_payment_date_key_unique),
    ("dim_payment_deadline_date", test_dim_payment_deadline_date_key_unique),
    ("dim_staff", test_dim_staff_staff_key_unique),
    ("dim_manager (manager_key)", test_dim_manager_manager_key_unique),
    ("dim_manager (staff_id)", test_dim_manager_staff_id_unique),
    ("dim_store (store_key)", test_dim_store_store_key_unique),
    ("dim_store (store_id)", test_dim_store_store_id_unique),
    ("dim_car", test_dim_car_car_key_unique),
    ("dim_customer (customer_key)", test_dim_customer_customer_key_unique),
    ("dim_customer (customer_id)", test_dim_customer_customer_id_unique),
    ("dim_equipment", test_dim_equipment_equipment_key_unique),

    # Bridge table tests
    ("bridge_car_equipment", test_bridge_car_equipment_car_key_unique),
    ("bridge_equipment_group (FK to car_equipment)", test_bridge_equipment_group_equipment_group_key_exists),
    ("bridge_equipment_group (FK to equipment)", test_bridge_equipment_group_equipment_key_exists),
    ("bridge_staff_hierarchy (FK to staff)", test_bridge_staff_hierarchy_staff_key_exists),
    ("bridge_staff_hierarchy (FK to manager)", test_bridge_staff_hierarchy_manager_key_exists),

    # Fact table tests
    ("fact_service (service_key)", test_fact_service_key_unique),
    ("fact_service (FK to car)", test_fact_service_car_key_exists),
    ("fact_rental (rental_key)", test_fact_rental_key_unique),
    ("fact_rental (FK to customer)", test_fact_rental_customer_key_exists),
    ("fact_rental (FK to car)", test_fact_rental_car_key_exists),
    ("fact_rental (FK to staff)", test_fact_rental_staff_key_exists),
    ("fact_rental (FK to store)", test_fact_rental_store_key_exists),
    ("fact_rental (FK to date)", test_fact_rental_date_keys_exist),

    # Required field completeness tests
    ("dim_customer (NOT NULL)", test_dim_customer_required_fields),
    ("dim_staff (NOT NULL)", test_dim_staff_required_fields),
    ("dim_store (NOT NULL)", test_dim_store_required_fields),
    ("dim_car (NOT NULL)", test_dim_car_required_fields),
    ("fact_rental (NOT NULL)", test_fact_rental_required_fields),
    ("fact_service (NOT NULL)", test_fact_service_required_fields),

    # Data reconciliation tests (ETL integrity)
    ("fact_rental (payment reconciliation)", test_fact_rental_payment_reconciliation),
]

passed = 0
failed = 0
failed_tests = []

for test_name, test_func in test_functions:
    try:
        logger.info(f"Running: {test_name} - {test_func.__doc__}")
        test_func()
        passed += 1
        logger.info(f"✅ PASS: {test_name}\n")
    except AssertionError as e:
        failed += 1
        failed_tests.append({
            "test": test_name,
            "description": test_func.__doc__,
            "error": str(e)
        })
        logger.error(f"❌ FAIL: {test_name}")
        logger.error(f"   {str(e)}\n")
    except Exception as e:
        failed += 1
        failed_tests.append({
            "test": test_name,
            "description": test_func.__doc__,
            "error": f"Unexpected error: {str(e)}"
        })
        logger.error(f"❌ ERROR: {test_name}")
        logger.error(f"   Unexpected error: {str(e)}\n")

total = len(test_functions)

logger.info("=" * 70)
logger.info("TEST EXECUTION SUMMARY")
logger.info("=" * 70)
logger.info(f"Total Tests: {total}")
logger.info(f"Passed: {passed} ✅")
logger.info(f"Failed: {failed} ❌")
logger.info(f"Success Rate: {(passed/total*100):.1f}%")

if failed > 0:
    logger.error("\n" + "=" * 70)
    logger.error("FAILED TESTS DETAILS")
    logger.error("=" * 70)
    for test in failed_tests:
        logger.error(f"\n❌ {test['test']}")
        logger.error(f"   Description: {test['description']}")
        logger.error(f"   Error: {test['error']}")
    logger.error("\n" + "=" * 70)
    logger.error(f"⚠️  {failed} TEST(S) FAILED - REVIEW REQUIRED")
    logger.error("=" * 70)

    raise Exception(f"Data quality tests failed: {failed}/{total}")
else:
    logger.info("\n" + "=" * 70)
    logger.info("🎉 ALL DATA QUALITY TESTS PASSED!")
    logger.info("=" * 70)
    logger.info("Coverage:")
    logger.info("  • Dimensions: 11 tables (date + 5 date role variants, staff, store, car, customer, equipment)")
    logger.info("  • Bridge tables: 3 tables (staff_hierarchy, car_equipment, equipment_group)")
    logger.info("  • Fact tables: 2 tables (service, rental)")
    logger.info("  • Required field completeness: 6 validation groups (NOT NULL checks)")
    logger.info("  • Data reconciliation: 1 test (payment amounts ETL integrity)")
    logger.info("=" * 70)

